## Middleware

##### MiddleWare provides a way to more tightly control what happens inside the agent. Middleware is useful for the following:


1. tracking agent behavior with logging, analytics and debugging

2. transforming prompt, tool selection, and output formatting

3. adding retires, fallbacks and early termination logic
    
4. applying rate limits, guardrails and PII(Personally Identifiable Information) detection

In [2]:
import os
from dotenv import load_dotenv
load_dotenv()

from langchain.chat_models import init_chat_model

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

### Summarization MiddleWare


##### automatically summarize conversation hustory when approaching token limits, preserving recent messages while compressing older context. Summarization is useful for the following:

    . Long-running conversations that exceed context windows.
    . Multi-Turn dialogues with extensive history.
    . applications where preserving full conversation context matters.

In [3]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage, SystemMessage

## messageBased summarization

agent=create_agent(
    model=init_chat_model("openai/gpt-oss-20b", model_provider = 'groq'),
    checkpointer = InMemorySaver(),
    middleware = [
        SummarizationMiddleware(
            model=init_chat_model("openai/gpt-oss-20b", model_provider = 'groq'),
            trigger =("messages",10),
            keep=("messages",4)
        )
            
    ]
)

In [8]:
### Run with thread id

config={"configurable":{"thread_id":"test-1"}}

In [13]:
# alt test data

questions = [
    "what is 2+2?",
    "what is 10*5?",
    "what is 100/4?",
    "what is 123-531?",
    "what is 999*3?",
    "what is 4*4?",
]

for q in questions:
    response = agent.invoke({"messages":[HumanMessage(content=q)]},config)
    print(f"Messages: {response}")
    print(f"Messages: {len(response['messages'])}")

Messages: {'messages': [HumanMessage(content='Here is a summary of the conversation to date:\n\n## SESSION INTENT\nThe user wants the results of simple arithmetic expressions.\n\n## SUMMARY\n- Expressions requested so far and their answers:\n  - 2\u202f+\u202f2\u202f=\u202f**4**\n  - 10\u202f×\u202f5\u202f=\u202f**50**\n  - 4\u202f×\u202f4\u202f=\u202f**16**\n  - 123\u202f−\u202f531\u202f=\u202f**–408**\n  - 999\u202f×\u202f3\u202f=\u202f**2\u202f997**\n- Pending expression:\n  - 100\u202f÷\u202f4 (the user just asked for this; it has not yet been answered)\n\n## ARTIFACTS\nNone.\n\n## NEXT STEPS\nCompute and provide the result of **100\u202f÷\u202f4**.', additional_kwargs={'lc_source': 'summarization'}, response_metadata={}, id='9e4aa256-0483-4fa4-915f-ca9e9f019124'), AIMessage(content='100\u202f÷\u202f4\xa0=\xa0**25**.', additional_kwargs={'reasoning_content': 'User is asking for simple arithmetic. Provide answer.'}, response_metadata={'token_usage': {'completion_tokens': 32, 'prompt

### summarization after tokens instead of len

In [16]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.tools import tool

@tool
def search_hotels(city: str) -> str:
    """search hotels - returns long response to use more tokens"""
    return f"""hotels in{city}:
    1. Grand Hotel - 5 star, $300/night, spa, pool, gym
    2. city inn - 4 star, #180/night, business center
    3. budget stay - 3 star, $75/night, free wifi"""

agent = create_agent(
    model=init_chat_model("openai/gpt-oss-20b", model_provider = 'groq'),
    tools = [search_hotels],
    checkpointer = InMemorySaver(),
    middleware = [
        SummarizationMiddleware(
            model=init_chat_model("openai/gpt-oss-20b", model_provider = 'groq'),
            trigger = ("tokens", 550),
            keep = ("tokens", 200),
        ),
    ]
)

config = {"configurable": {"thread_id": "test-1"}}

#token counter (approx)

def count_tokens(messages):
    total_chars = sum(len(str(m.content)) for m in messages)
    return total_chars // 4 ## 4 chars = 1 token

In [17]:
# run test

cities = ["paris", "london", "tokyo", "new york", "dubai", "singapore"]

for city in cities:
    response = agent.invoke(
        {"messages": [HumanMessage(content=f"Find hotels in {city}")]},
        config = config
    )


    tokens = count_tokens(response["messages"])
    print(f"{city}: ~{tokens} tokens, {len(response['messages'])} messages")
    print(f"{(response['messages'])}")

paris: ~215 tokens, 4 messages
[HumanMessage(content='Find hotels in paris', additional_kwargs={}, response_metadata={}, id='70b21822-eb81-4114-95b3-52d41292fa51'), AIMessage(content='', additional_kwargs={'reasoning_content': 'We need to call the function search_hotels with city="Paris".', 'tool_calls': [{'id': 'fc_8fb51789-412e-4752-b067-d03ee3cbe07d', 'function': {'arguments': '{"city":"Paris"}', 'name': 'search_hotels'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 39, 'prompt_tokens': 129, 'total_tokens': 168, 'completion_time': 0.040496763, 'completion_tokens_details': {'reasoning_tokens': 15}, 'prompt_time': 0.007235562, 'prompt_tokens_details': None, 'queue_time': 0.043651737, 'total_time': 0.047732325}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_80501ff3a1', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a1349b-af94-7da1-bc86-d9ab50d5985a-0', tool_calls=

### Fraction Based

In [4]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langgraph.checkpoint.memory import InMemorySaver

@tool
def search_hotels(city: str) -> str:
    """Search hotels."""
    return f"Hotels in {city}: Grand Hotel $350, City Inn $180, Budget Stay $75"

# LOW fraction for testing!
agent = create_agent(
    model=init_chat_model("openai/gpt-oss-20b", model_provider = 'groq'),
    tools=[search_hotels],
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model=init_chat_model("openai/gpt-oss-20b", model_provider = 'groq'),
            trigger=("fraction", 0.005),  # 0.5% = ~640 tokens
            keep=("fraction", 0.002),     # 0.2% = ~256 tokens
        ),
    ],
)

config = {"configurable": {"thread_id": "test-1"}}

# Token counter
def count_tokens(messages):
    return sum(len(str(m.content)) for m in messages) // 4

# Test
cities = ["Paris", "London", "Tokyo", "New York", "Dubai", "Singapore"]

for city in cities:
    response = agent.invoke(
        {"messages": [HumanMessage(content=f"Hotels in {city}")]},
        config=config
    )
    tokens = count_tokens(response["messages"])
    fraction = tokens / 128000  # gpt-4o-mini context
    print(f"{city}: ~{tokens} tokens ({fraction:.4%}), {len(response['messages'])} msgs")
    print(response['messages'])

Paris: ~136 tokens (0.1063%), 4 msgs
[HumanMessage(content='Hotels in Paris', additional_kwargs={}, response_metadata={}, id='a649de7d-24a9-4d5a-b51e-7b18ec184525'), AIMessage(content='', additional_kwargs={'reasoning_content': 'The user says "Hotels in Paris". They likely want a search for hotels. We have a function search_hotels(city: string). We\'ll call it with city "Paris". Then we provide results. Probably we want to call the function.', 'tool_calls': [{'id': 'fc_6ab464d5-fd2c-4639-b1b6-d43817a899cc', 'function': {'arguments': '{"city":"Paris"}', 'name': 'search_hotels'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 73, 'prompt_tokens': 120, 'total_tokens': 193, 'completion_time': 0.077208611, 'completion_tokens_details': {'reasoning_tokens': 49}, 'prompt_time': 0.005714549, 'prompt_tokens_details': None, 'queue_time': 0.398083687, 'total_time': 0.08292316}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_f102a3179c', 'service_tier'

## Human In the Loop MiddleWare
##### Pause agent execution for human approval, editing, or rejection of tool calls before they execute. Human-in-the-loop is useful for the following:

1. High-stakes operations requiring human approval (e.g. database writes, financial transactions).

2. Compliance workflows where human oversight is mandatory.

3. Long-running conversations where human feedback guides the agent.

In [16]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver

def read_email_tool(email_id: str) -> str:
    """Mock function to read an email by its ID."""
    return f"Email content for ID: {email_id}"

def send_email_tool(recipient: str, subject: str, body: str) -> str:
    """Mock function to send an email."""
    return f"Email sent to {recipient} with subject '{subject}'"

In [ ]:
agent=create_agent(
    model=init_chat_model("openai/gpt-oss-20b", model_provider = 'groq'),
    tools=[read_email_tool,send_email_tool],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email_tool":{
                    "allowed_decisions":["approve","edit","reject"]
                },
                "read_email_tool":False,

            }
        )
    ]
)

In [ ]:
config = {"configurable": {"thread_id": "test-approve"}}
# Step 1: Request
result = agent.invoke(
    {"messages": [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'")]},
    config=config
)

In [19]:
result

{'messages': [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'", additional_kwargs={}, response_metadata={}, id='dd54376f-2fe0-4995-ab5d-5b5696460908'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'We need to call send_email_tool.', 'tool_calls': [{'id': 'fc_f4b511e9-cc95-4346-8371-75ede085f91b', 'function': {'arguments': '{"body":"How are you?","recipient":"john@test.com","subject":"Hello"}', 'name': 'send_email_tool'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 46, 'prompt_tokens': 174, 'total_tokens': 220, 'completion_time': 0.055475387, 'completion_tokens_details': {'reasoning_tokens': 9}, 'prompt_time': 0.01631469, 'prompt_tokens_details': None, 'queue_time': 0.31711734, 'total_time': 0.071790077}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_4f7e7dc26e', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='l

In [20]:
from langgraph.types import Command
# Step 2: Approve
if "__interrupt__" in result:
    print("⏸️ Paused! Approving...")
    
    result = agent.invoke(
        Command(
            resume={
                "decisions": [
                    {"type": "approve"}
                ]
            }
        ),
        config=config
    )
    
    print(f"✅ Result: {result['messages'][-1].content}")

⏸️ Paused! Approving...
✅ Result: ✅ Email sent to john@test.com with subject “Hello”.


In [21]:
result

{'messages': [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'", additional_kwargs={}, response_metadata={}, id='dd54376f-2fe0-4995-ab5d-5b5696460908'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'We need to call send_email_tool.', 'tool_calls': [{'id': 'fc_f4b511e9-cc95-4346-8371-75ede085f91b', 'function': {'arguments': '{"body":"How are you?","recipient":"john@test.com","subject":"Hello"}', 'name': 'send_email_tool'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 46, 'prompt_tokens': 174, 'total_tokens': 220, 'completion_time': 0.055475387, 'completion_tokens_details': {'reasoning_tokens': 9}, 'prompt_time': 0.01631469, 'prompt_tokens_details': None, 'queue_time': 0.31711734, 'total_time': 0.071790077}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_4f7e7dc26e', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='l

### for Rejection

In [23]:
agent=create_agent(
    model=init_chat_model("openai/gpt-oss-20b", model_provider = 'groq'),
    tools=[read_email_tool,send_email_tool],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email_tool":{
                    "allowed_decisions":["approve","edit","reject"]
                },
                "read_email_tool":False,

            }
        )
    ]
)

In [24]:
config = {"configurable": {"thread_id": "test-approve"}}
# Step 1: Request
result = agent.invoke(
    {"messages": [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'")]},
    config=config
)

In [25]:
from langgraph.types import Command
# Step 2: Approve
if "__interrupt__" in result:
    print("⏸️ Paused! Approving...")
    
    result = agent.invoke(
        Command(
            resume={
                "decisions": [
                    {"type": "reject"}
                ]
            }
        ),
        config=config
    )
    
    print(f"✅ Result: {result['messages'][-1].content}")

⏸️ Paused! Approving...
✅ Result: I’m sorry, but I can’t send the email right now. If you’d like me to try again, just let me know!


In [26]:
result

{'messages': [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'", additional_kwargs={}, response_metadata={}, id='c55551b9-34ac-4afc-a00f-1ed9b7b7a70c'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'We need to call the send_email_tool. Use function.', 'tool_calls': [{'id': 'fc_3422bf5f-9890-4b1c-bfc9-6067a2825944', 'function': {'arguments': '{"body":"How are you?","recipient":"john@test.com","subject":"Hello"}', 'name': 'send_email_tool'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 50, 'prompt_tokens': 174, 'total_tokens': 224, 'completion_time': 0.067246665, 'completion_tokens_details': {'reasoning_tokens': 13}, 'prompt_time': 0.012204096, 'prompt_tokens_details': None, 'queue_time': 0.334314802, 'total_time': 0.079450761}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_2b688e7cc3', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_prov

## Editing

In [27]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver

def read_email_tool(email_id: str) -> str:
    """Mock function to read an email by its ID."""
    return f"Email content for ID: {email_id}"

def send_email_tool(recipient: str, subject: str, body: str) -> str:
    """Mock function to send an email."""
    return f"Email sent to {recipient} with subject '{subject}'"

In [28]:
agent=create_agent(
    model=init_chat_model("openai/gpt-oss-20b", model_provider = 'groq'),
    tools=[read_email_tool,send_email_tool],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email_tool":{
                    "allowed_decisions":["approve","edit","reject"]
                },
                "read_email_tool":False,

            }
        )
    ]
)

In [29]:
config = {"configurable": {"thread_id": "test-edit"}}

# Step 1: Request (with wrong info)
result = agent.invoke(
    {"messages": [HumanMessage(content="Send email to wrong@email.com with subject 'Test' and body 'Hello'")]},
    config=config
)

In [30]:
result

{'messages': [HumanMessage(content="Send email to wrong@email.com with subject 'Test' and body 'Hello'", additional_kwargs={}, response_metadata={}, id='41a17fdc-2060-47f5-bf4c-f167630e701c'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'We need to call send_email_tool.', 'tool_calls': [{'id': 'fc_4c302f0a-0b57-4057-8d48-20f5c2b17b83', 'function': {'arguments': '{"body":"Hello","recipient":"wrong@email.com","subject":"Test"}', 'name': 'send_email_tool'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 43, 'prompt_tokens': 172, 'total_tokens': 215, 'completion_time': 0.060022234, 'completion_tokens_details': {'reasoning_tokens': 9}, 'prompt_time': 0.008449799, 'prompt_tokens_details': None, 'queue_time': 0.337881687, 'total_time': 0.068472033}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_334cc21c60', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a

In [31]:
# Step 2: Edit and approve
if "__interrupt__" in result:
    print("⏸️ Paused! Editing...")
    
    result = agent.invoke(
        Command(
            resume={
                "decisions": [
                    {
                        "type": "edit",
                        "edited_action": {
                            "name": "send_email_tool",      # Tool name
                            "args": {                   # New arguments
                                "recipient": "correct@email.com",
                                "subject": "Corrected Subject",
                                "body": "This was edited by human before sending"
                            }
                        }
                    }
                ]
            }
        ),
        config=config
    )
    
    print(f"✏️ Result: {result['messages'][-1].content}")

⏸️ Paused! Editing...
✏️ Result: Your email has been sent.


In [32]:
result

{'messages': [HumanMessage(content="Send email to wrong@email.com with subject 'Test' and body 'Hello'", additional_kwargs={}, response_metadata={}, id='41a17fdc-2060-47f5-bf4c-f167630e701c'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'We need to call send_email_tool.', 'tool_calls': [{'id': 'fc_4c302f0a-0b57-4057-8d48-20f5c2b17b83', 'function': {'arguments': '{"body":"Hello","recipient":"wrong@email.com","subject":"Test"}', 'name': 'send_email_tool'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 43, 'prompt_tokens': 172, 'total_tokens': 215, 'completion_time': 0.060022234, 'completion_tokens_details': {'reasoning_tokens': 9}, 'prompt_time': 0.008449799, 'prompt_tokens_details': None, 'queue_time': 0.337881687, 'total_time': 0.068472033}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_334cc21c60', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a